In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# 1. Load and explore dataset
df = pd.read_csv("telecom_churn_dataset.csv")
print(df.head())
df.info()
print(df.isnull().sum())

   customer_id  gender  age  senior_citizen           state city_tier  \
0  CUST-000001    Male   26               0  Andhra Pradesh    Tier 3   
1  CUST-000002  Female   53               0      Tamil Nadu    Tier 1   
2  CUST-000003  Female   23               0       Rajasthan    Tier 3   
3  CUST-000004  Female   41               0       Karnataka    Tier 2   
4  CUST-000005    Male   43               0          Kerala    Tier 3   

   tenure_months   contract_type plan_type     data_plan  ... total_day_calls  \
0             19  Month-to-Month  Postpaid          1 GB  ...              14   
1             61        One Year   Prepaid          5 GB  ...              82   
2             28        One Year  Postpaid          5 GB  ...              17   
3             56  Month-to-Month   Prepaid          2 GB  ...               3   
4              1        Two Year  Postpaid  No Data Plan  ...               3   

  total_eve_calls total_night_calls total_intl_calls customer_service_call

In [3]:

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower().replace("", pd.NA)

In [4]:
# 3. Remove duplicates
df = df.drop_duplicates()

In [5]:
# 4. Handle missing values
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

In [6]:

# 5. Handle outliers using IQR
num_cols = [
    "age", "tenure_months", "data_usage_gb",
    "total_day_minutes", "total_eve_minutes",
    "total_night_minutes", "total_intl_minutes",
    "total_day_calls", "total_eve_calls",
    "total_night_calls", "total_intl_calls",
    "customer_service_calls", "monthly_charges", "total_charges"
]

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

In [7]:

# 6. Feature engineering
df["tenure_group"] = pd.cut(
    df["tenure_months"],
    bins=[0, 12, 36, 60, 72],
    labels=["0-12", "13-36", "37-60", "61-72"]
)

In [8]:
# 7. Encode target and categorical columns
df["churn"] = df["churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
df = pd.get_dummies(df.drop(columns=["customer_id"]), drop_first=True)

In [9]:
# 8. Save cleaned dataset
df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)

In [10]:
# 9. Split into training and testing data
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 10. Scale numerical features
scaler = StandardScaler()
X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
X_test.loc[:, num_cols] = scaler.transform(X_test[num_cols])

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Cleaned dataset saved!")

Training shape: (40000, 63)
Testing shape: (10000, 63)
Cleaned dataset saved!


C:\Users\Manas\AppData\Local\Temp\ipykernel_21940\1681113390.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.1485159  -1.46976498 -1.34965143 ...  1.71324416  0.03165443
  1.23278995]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
C:\Users\Manas\AppData\Local\Temp\ipykernel_21940\1681113390.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.61007556  0.46551874  0.65826116 ...  0.36914752 -0.01633733
  0.80281798]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
C:\Users\Manas\AppData\Local\Temp\ipykernel_21940\1681113390.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a f